# Step-by-step: Metric Evaluation of Monocular Depth Estimation (ArUco Plane Geometry)

This notebook walks through the full pipeline implemented in `Metric_Evaluation_Monocular_Depth_Estimation/src/`:

1. Load camera calibration (intrinsics + distortion)
2. Load an evaluation image (captured with the calibrated webcam)
3. Detect ArUco marker + estimate pose (solvePnP) → metric ground-truth plane
4. Predict monocular depth (default backend: MiDaS)
5. Backproject depth inside the marker region → predicted 3D points
6. Fit a plane to predicted points, align scale to the metric plane, compute metrics
7. Render an overlay visualization

> Note: Capturing chessboard images and capturing an evaluation image require an interactive webcam window.
> This notebook focuses on analyzing an existing image in `data/images/`.


In [ ]:
from pathlib import Path
import sys

# Project root = Metric_Evaluation_Monocular_Depth_Estimation/
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

# If you started Jupyter from repo root, the project root is:
if (PROJECT_ROOT / 'Metric_Evaluation_Monocular_Depth_Estimation').exists():
    PROJECT_ROOT = PROJECT_ROOT / 'Metric_Evaluation_Monocular_Depth_Estimation'

SRC_ROOT = PROJECT_ROOT / 'src'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('PROJECT_ROOT:', PROJECT_ROOT)
print('SRC_ROOT:', SRC_ROOT)
print('Python path ok:', str(PROJECT_ROOT) in sys.path)


In [ ]:
import numpy as np
import cv2

print('OpenCV version:', cv2.__version__)
print('Has cv2.aruco:', hasattr(cv2, 'aruco'))


## 1) Load camera calibration

Calibration files are generated by:

- `python -m src.calibration.capture_chessboard`
- `python -m src.calibration.calibrate_camera`

They are expected at:

- `data/calibration/camera_matrix.yaml`
- `data/calibration/dist_coeffs.yaml`


In [ ]:
from src.detection.aruco_pose import load_calibration

cam_path = PROJECT_ROOT / 'data/calibration/camera_matrix.yaml'
dist_path = PROJECT_ROOT / 'data/calibration/dist_coeffs.yaml'

assert cam_path.exists(), f"Missing: {cam_path}"
assert dist_path.exists(), f"Missing: {dist_path}"

K, dist = load_calibration(str(cam_path), str(dist_path))
print('K=\n', K)
print('dist=', dist.reshape(-1))


## 2) Load evaluation image

Either:
- capture one via CLI: `python -m src.main --capture --capture_output data/images/capture.jpg`
- or place any image captured with the **same calibrated webcam** into `data/images/`


In [ ]:
import matplotlib.pyplot as plt

img_dir = PROJECT_ROOT / 'data/images'
assert img_dir.exists(), f"Missing folder: {img_dir}"

# Use a specific path if you want:
image_path = img_dir / 'capture.jpg'
if not image_path.exists():
    imgs = sorted(list(img_dir.glob('*.jpg')) + list(img_dir.glob('*.png')))
    assert imgs, f"No images found in {img_dir}"
    image_path = imgs[-1]

bgr = cv2.imread(str(image_path))
assert bgr is not None, f"Could not read image: {image_path}"

rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8,5))
plt.imshow(rgb)
plt.title(f"Evaluation image: {image_path.name}")
plt.axis('off')
plt.show()


## 3) Detect ArUco marker, estimate pose, compute metric ground-truth plane

We use:
- `cv2.aruco.ArucoDetector` for detection
- `cv2.solvePnP` (IPPE_SQUARE if available) for pose

Ground-truth plane in camera coordinates:

- plane equation: `n_gt^T X + d_gt = 0`


In [ ]:
from src.detection.aruco_pose import detect_aruco_pose

marker_length_m = 0.048  # adjust to your measured marker side length
marker_id = None         # set to an int (e.g. 0) to force a specific marker id

det = detect_aruco_pose(
    bgr=bgr,
    camera_matrix=K,
    dist_coeffs=dist,
    marker_length_m=marker_length_m,
    marker_id=marker_id,
)
assert det is not None, 'No marker detected.'

print('Detected marker id:', det.marker_id)
print('rvec:', det.rvec)
print('tvec (m):', det.tvec)
print('n_gt:', det.n_gt)
print('d_gt:', det.d_gt)


In [ ]:
# Visualize marker corners and mask
vis = bgr.copy()
pts = det.corners_px.reshape(4,2).astype(int)
import cv2
cv2.polylines(vis, [pts], True, (0,255,0), 2)

mask_vis = cv2.cvtColor(det.mask, cv2.COLOR_GRAY2RGB)
mask_vis = (mask_vis * 0.6 + cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB) * 0.4).astype(np.uint8)

plt.figure(figsize=(12,5))
plt.subplot(1,2,1)
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('ArUco polygon overlay')
plt.axis('off')

plt.subplot(1,2,2)
plt.imshow(mask_vis)
plt.title('Marker region mask (alpha overlay)')
plt.axis('off')
plt.show()


## 4) Predict monocular depth (MiDaS backend)

The depth backend is **pluggable**. By default the project uses MiDaS via `torch.hub`.

The output depth is **relative** (unknown scale), so we align it using the metric plane constraint.


In [ ]:
from src.depth.factory import create_depth_backend

depth_backend = 'midas'  # midas | depth_anything_v2 (stub)
model = create_depth_backend(depth_backend)

pred = model.predict(bgr)
depth_rel = pred.depth
print('Depth meta:', pred.meta)
print('Depth shape:', depth_rel.shape, 'min/max:', float(depth_rel.min()), float(depth_rel.max()))


In [ ]:
plt.figure(figsize=(8,5))
plt.imshow(depth_rel, cmap='inferno')
plt.title('Relative depth (visualization)')
plt.axis('off')
plt.colorbar(fraction=0.046, pad=0.04)
plt.show()


## 5) Backproject depth inside marker region → 3D points

Using intrinsics:

- `X = (u - cx)/fx * Z`
- `Y = (v - cy)/fy * Z`
- `Z = depth(u, v)`

(Depth is still relative.)


In [ ]:
from src.depth.backprojection import Intrinsics, backproject_depth_to_points

intr = Intrinsics.from_camera_matrix(K)
pts_rel = backproject_depth_to_points(depth_rel, intr, mask=det.mask)
print('Backprojected points:', pts_rel.shape)


## 6) Plane fit + scale alignment + metrics

We fit a plane to the predicted points (SVD), then compute a scale `s` such that the scaled points best satisfy the GT plane.

Finally we compute:
- normal angle error (deg)
- plane offset error (m)
- RMSE point-to-plane distance (m)


In [ ]:
from src.depth.plane_alignment import fit_plane_svd, align_points_to_plane
from src.evaluation.metrics import compute_metrics

plane_rel = fit_plane_svd(pts_rel)
pts_metric, s = align_points_to_plane(pts_rel, det.n_gt, det.d_gt)
plane_metric = fit_plane_svd(pts_metric)

metrics = compute_metrics(
    n_gt=det.n_gt,
    d_gt=det.d_gt,
    n_pred=plane_metric.n,
    d_pred=plane_metric.d,
    points_metric=pts_metric,
)

print('scale_factor_s:', s)
print('normal_angle_deg:', metrics.normal_angle_deg)
print('plane_offset_abs_m:', metrics.plane_offset_abs_m)
print('rmse_point_to_gt_plane_m:', metrics.rmse_point_to_gt_plane_m)


## 7) Overlay visualization + save result

We reuse the visualization helper from the project.


In [ ]:
from src.visualization.draw_pose import draw_aruco_overlay

out = draw_aruco_overlay(
    bgr=bgr,
    corners_px=det.corners_px,
    rvec=det.rvec,
    tvec=det.tvec,
    camera_matrix=K,
    dist_coeffs=dist,
    metrics=metrics,
)

out_path = PROJECT_ROOT / 'outputs/visualizations/notebook_overlay.jpg'
out_path.parent.mkdir(parents=True, exist_ok=True)
cv2.imwrite(str(out_path), out)

plt.figure(figsize=(10,6))
plt.imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
plt.title(f'Overlay saved to: {out_path.name}')
plt.axis('off')
plt.show()


## Next steps

- Run a **distance sweep** or **angle sweep** by capturing multiple images and looping over them.
- Integrate **Depth Anything V2** by implementing `src/depth/depth_anything_v2_backend.py`.
